# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [12]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [13]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

## Data Registration

In [14]:
os.makedirs("tourism_project/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

In [17]:
%%writefile tourism_project/model_building/data_register.py
import os
from getpass import getpass
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter HF_TOKEN: ").strip()

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is still empty.")

REPO_ID = "SachinY1996/visit-with-us-tourism-prediction"
REPO_TYPE = "dataset"
LOCAL_DATA_DIR = "tourism_project/data"

api = HfApi(token=HF_TOKEN)

try:
    api.repo_info(repo_id=REPO_ID, repo_type=REPO_TYPE)
    print(f"Dataset repo '{REPO_ID}' already exists.")
except RepositoryNotFoundError:
    create_repo(repo_id=REPO_ID, repo_type=REPO_TYPE, private=False, token=HF_TOKEN)
    print(f"Created dataset repo '{REPO_ID}'.")

api.upload_folder(
    folder_path=LOCAL_DATA_DIR,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    commit_message="Upload raw tourism dataset"
)

print("Raw dataset uploaded successfully.")

Overwriting tourism_project/model_building/data_register.py


### Execute Data Registration Script

Run the cell below to execute the `data_register.py` script. This will create your Hugging Face dataset repository if it doesn't already exist and upload the initial `tourism.csv` file.

In [18]:
!python tourism_project/model_building/data_register.py

Enter HF_TOKEN: 
Dataset repo 'Nikidsouza23/visit-with-us-tourism-prediction' already exists.
No files have been modified since last commit. Skipping to prevent empty commit.
Raw dataset uploaded successfully.


## Data Preparation

In [21]:
%%writefile tourism_project/model_building/prep.py
import os
from getpass import getpass
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter HF_TOKEN: ").strip()

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is still empty.")

DATASET_REPO = "SachinY1996/visit-with-us-tourism-prediction"
LOCAL_DATA_PATH = "tourism_project/data/tourism.csv"
OUTPUT_DIR = "tourism_project/data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
api = HfApi(token=HF_TOKEN)

try:
    api.repo_info(repo_id=DATASET_REPO, repo_type="dataset")
except RepositoryNotFoundError:
    create_repo(repo_id=DATASET_REPO, repo_type="dataset", private=False, token=HF_TOKEN)

df = pd.read_csv(LOCAL_DATA_PATH)
print("Dataset loaded successfully.")
print("Original shape:", df.shape)

unnamed_cols = [c for c in df.columns if "Unnamed" in str(c)]
if unnamed_cols:
    df.drop(columns=unnamed_cols, inplace=True)

df.drop(columns=["CustomerID"], inplace=True, errors="ignore")
df.drop_duplicates(inplace=True)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

replacements = {
    "Gender": {"Fe Male": "Female"},
    "Occupation": {"Free Lancer": "Freelancer"},
    "TypeofContact": {"Self Inquiry": "Self Enquiry"},
    "MaritalStatus": {"Unmarried": "Single"},
}
for col, mapping in replacements.items():
    if col in df.columns:
        df[col] = df[col].replace(mapping)

for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    mode_val = df[col].mode(dropna=True)
    if not mode_val.empty:
        df[col] = df[col].fillna(mode_val.iloc[0])

target_col = "ProdTaken"
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

paths = {
    "Xtrain.csv": os.path.join(OUTPUT_DIR, "Xtrain.csv"),
    "Xtest.csv": os.path.join(OUTPUT_DIR, "Xtest.csv"),
    "ytrain.csv": os.path.join(OUTPUT_DIR, "ytrain.csv"),
    "ytest.csv": os.path.join(OUTPUT_DIR, "ytest.csv"),
}

X_train.to_csv(paths["Xtrain.csv"], index=False)
X_test.to_csv(paths["Xtest.csv"], index=False)
y_train.to_frame(name=target_col).to_csv(paths["ytrain.csv"], index=False)
y_test.to_frame(name=target_col).to_csv(paths["ytest.csv"], index=False)

for remote_name, local_path in paths.items():
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=remote_name,
        repo_id=DATASET_REPO,
        repo_type="dataset",
        commit_message=f"Upload {remote_name}"
    )

print("Train/test splits uploaded successfully.")

Overwriting tourism_project/model_building/prep.py


### Run Data Preparation Script

Run the cell below to execute the `prep.py` script. This will prepare the data and upload the `Xtrain.csv`, `Xtest.csv`, `ytrain.csv`, and `ytest.csv` files to your Hugging Face dataset repository.

In [22]:
!python tourism_project/model_building/prep.py

Enter HF_TOKEN: 
Dataset loaded successfully.
Original shape: (4128, 21)
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
Train/test splits uploaded successfully.


## Model Training and Registration with Experimentation Tracking

In [24]:
import subprocess
import mlflow

# Set your auth token here (replace with your actual token)

# Start MLflow UI on port 5000

# Create public tunnel
print("MLflow UI is available at:", public_url)

MLflow UI is available at: https://xerox-sprig-yo-yo.ngrok-free.dev


In [25]:
# Set the tracking URL for MLflow
mlflow.set_tracking_uri(public_url)

# Set the name for the experiment
mlflow.set_experiment("visit-with-us-tourism-classification")

<Experiment: artifact_location='mlflow-artifacts:/997063706620704190', creation_time=1782027543117, experiment_id='997063706620704190', last_update_time=1782027543117, lifecycle_stage='active', name='visit-with-us-tourism-classification', tags={}>

In [26]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib
import mlflow
from huggingface_hub import HfApi

# Define Hugging Face dataset repository and token
HF_DATASET_REPO = "SachinY1996/visit-with-us-tourism-prediction"
HF_TOKEN = os.getenv("HF_TOKEN")
api = HfApi(token=HF_TOKEN)

# Define paths to the split datasets on Hugging Face
Xtrain_path = f"hf://datasets/{HF_DATASET_REPO}/Xtrain.csv"
Xtest_path  = f"hf://datasets/{HF_DATASET_REPO}/Xtest.csv"
ytrain_path = f"hf://datasets/{HF_DATASET_REPO}/ytrain.csv"
ytest_path  = f"hf://datasets/{HF_DATASET_REPO}/ytest.csv"

# Load data directly from Hugging Face
X_train = pd.read_csv(Xtrain_path)
X_test = pd.read_csv(Xtest_path)
y_train = pd.read_csv(ytrain_path).squeeze()
y_test = pd.read_csv(ytest_path).squeeze()

print("Train and test datasets loaded successfully from Hugging Face.")

# Dynamically determine numeric and categorical features
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

# Preprocessing pipelines for numeric and categorical features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Define base XGBoost Classifier
xgb_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Hyperparameter grid for XGBClassifier
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

# Create the full pipeline with preprocessor and XGBClassifier
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

with mlflow.start_run():
    # Grid Search
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=3, n_jobs=-1, scoring='f1', verbose=1)
    grid_search.fit(X_train, y_train)

    # Log parameter sets from GridSearchCV results
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]

        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_f1_score", mean_score)

    # Best model
    mlflow.log_params(grid_search.best_params_)
    best_model = grid_search.best_estimator_

    # Predictions
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    # Metrics for classification
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    # Log metrics
    mlflow.log_metrics(metrics)

    # Save and log the best model artifact
    model_path = "best_tourism_model.joblib"
    joblib.dump(best_model, model_path)
    mlflow.log_artifact(model_path)

    print("Model training and logging complete.")


Train and test datasets loaded successfully from Hugging Face.
Fitting 3 folds for each of 48 candidates, totalling 144 fits
🏃 View run ambitious-ant-391 at: https://xerox-sprig-yo-yo.ngrok-free.dev/#/experiments/997063706620704190/runs/a380b721049e45d0b1359da571d6d6c2
🧪 View experiment at: https://xerox-sprig-yo-yo.ngrok-free.dev/#/experiments/997063706620704190
🏃 View run adorable-fish-745 at: https://xerox-sprig-yo-yo.ngrok-free.dev/#/experiments/997063706620704190/runs/9910ae92510d45c8b3c032a97c700034
🧪 View experiment at: https://xerox-sprig-yo-yo.ngrok-free.dev/#/experiments/997063706620704190
🏃 View run clean-smelt-975 at: https://xerox-sprig-yo-yo.ngrok-free.dev/#/experiments/997063706620704190/runs/a64a6d79dd384c438798c8ab622c2ec0
🧪 View experiment at: https://xerox-sprig-yo-yo.ngrok-free.dev/#/experiments/997063706620704190
🏃 View run whimsical-hawk-906 at: https://xerox-sprig-yo-yo.ngrok-free.dev/#/experiments/997063706620704190/runs/718439d677b14bdfb5237581a27e3191
🧪 View e

### Experimentation and Tracking (Production Environment)

In [27]:
%%writefile tourism_project/model_building/train.py
import os
import joblib
import mlflow
import pandas as pd
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not set.")

HF_DATASET_REPO = "SachinY1996/visit-with-us-tourism-prediction"
HF_MODEL_REPO = "SachinY1996/visit-with-us-tourism-model"
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "file:./mlruns")
MODEL_LOCAL_PATH = "best_tourism_model.joblib"

api = HfApi(token=HF_TOKEN)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("visit-with-us-tourism-classification")

X_train = pd.read_csv(f"hf://datasets/{HF_DATASET_REPO}/Xtrain.csv")
X_test = pd.read_csv(f"hf://datasets/{HF_DATASET_REPO}/Xtest.csv")
y_train = pd.read_csv(f"hf://datasets/{HF_DATASET_REPO}/ytrain.csv").squeeze("columns")
y_test = pd.read_csv(f"hf://datasets/{HF_DATASET_REPO}/ytest.csv").squeeze("columns")

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0]
}

with mlflow.start_run(run_name="xgboost_gridsearch"):
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=3,
        scoring="f1",
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }

    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metrics(metrics)

    joblib.dump(best_model, MODEL_LOCAL_PATH)
    mlflow.log_artifact(MODEL_LOCAL_PATH, artifact_path="model")

    print("Best params:", grid_search.best_params_)
    print("Metrics:", metrics)

try:
    api.repo_info(repo_id=HF_MODEL_REPO, repo_type="model")
except RepositoryNotFoundError:
    create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=False, token=HF_TOKEN)

api.upload_file(
    path_or_fileobj=MODEL_LOCAL_PATH,
    path_in_repo="best_tourism_model.joblib",
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    commit_message="Upload best tourism model"
)

print(f"Model uploaded successfully to {HF_MODEL_REPO}")

Overwriting tourism_project/model_building/train.py


# Deployment

## Dockerfile

In [28]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [29]:
%%writefile tourism_project/deployment/Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Overwriting tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [41]:
%%writefile tourism_project/deployment/app.py
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

MODEL_REPO_ID = "SachinY1996/visit-with-us-tourism-model"
MODEL_FILENAME = "best_tourism_model.joblib"

@st.cache_resource
def load_model():
    model_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename=MODEL_FILENAME)
    return joblib.load(model_path)

model = load_model()

st.set_page_config(page_title="Visit with Us Predictor", layout="centered")
st.title("Wellness Tourism Package Prediction")
st.write("Predict whether a customer is likely to purchase the package before contacting them.")

age = st.number_input("Age", min_value=18, max_value=100, value=35)
typeofcontact = st.selectbox("Type of Contact", ["Company Invited", "Self Enquiry"])
occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Large Business", "Freelancer"])
gender = st.selectbox("Gender", ["Male", "Female"])
numberofpersonvisiting = st.number_input("Number of Person Visiting", min_value=1, max_value=20, value=2)
preferredpropertystar = st.number_input("Preferred Property Star", min_value=1.0, max_value=5.0, value=3.0)
maritalstatus = st.selectbox("Marital Status", ["Single", "Married", "Divorced"])
numberoftrips = st.number_input("Number of Trips", min_value=0.0, max_value=20.0, value=2.0)
passport = st.selectbox("Passport", [0, 1])
owncar = st.selectbox("Own Car", [0, 1])
numberofchildrenvisiting = st.number_input("Number of Children Visiting", min_value=0.0, max_value=10.0, value=0.0)
designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
monthlyincome = st.number_input("Monthly Income", min_value=1000.0, value=25000.0)
pitchsatisfactionscore = st.number_input("Pitch Satisfaction Score", min_value=1.0, max_value=5.0, value=3.0)
productpitched = st.selectbox("Product Pitched", ["Basic", "Deluxe", "Standard", "Super Deluxe", "King"])
numberoffollowups = st.number_input("Number of Followups", min_value=0.0, max_value=10.0, value=3.0)
durationofpitch = st.number_input("Duration of Pitch", min_value=1.0, max_value=100.0, value=15.0)
citytier = st.selectbox("City Tier", [1, 2, 3])

input_df = pd.DataFrame([{
    "Age": age,
    "TypeofContact": typeofcontact,
    "CityTier": citytier,
    "Occupation": occupation,
    "Gender": gender,
    "NumberOfPersonVisiting": numberofpersonvisiting,
    "PreferredPropertyStar": preferredpropertystar,
    "MaritalStatus": maritalstatus,
    "NumberOfTrips": numberoftrips,
    "Passport": passport,
    "OwnCar": owncar,
    "NumberOfChildrenVisiting": numberofchildrenvisiting,
    "Designation": designation,
    "MonthlyIncome": monthlyincome,
    "PitchSatisfactionScore": pitchsatisfactionscore,
    "ProductPitched": productpitched,
    "NumberOfFollowups": numberoffollowups,
    "DurationOfPitch": durationofpitch
}])

if st.button("Predict"):
    pred = model.predict(input_df)[0]
    prob = model.predict_proba(input_df)[0][1]

    if pred == 1:
        st.success("Prediction: Likely to Purchase")
    else:
        st.warning("Prediction: Not Likely to Purchase")

    st.info(f"Predicted purchase probability: {prob:.2%}")
    st.dataframe(input_df)

Overwriting tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [42]:
%%writefile tourism_project/deployment/requirements.txt
streamlit==1.43.2
pandas==2.2.2
scikit-learn==1.6.0

huggingface_hub==0.34.4
joblib==1.4.2
fsspec>=2024.6.1

Overwriting tourism_project/deployment/requirements.txt


# Hosting

In [43]:
os.makedirs("tourism_project/hosting", exist_ok=True)

In [44]:
%%writefile tourism_project/hosting/hosting.py
import os
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not set.")

SPACE_REPO = "SachinY1996/visit-with-us-tourism-app"
SPACE_SDK = "streamlit"
LOCAL_DEPLOYMENT_DIR = "tourism_project/deployment"

api = HfApi(token=HF_TOKEN)

try:
    api.repo_info(repo_id=SPACE_REPO, repo_type="space")
    print(f"Space '{SPACE_REPO}' already exists.")
except RepositoryNotFoundError:
    create_repo(
        repo_id=SPACE_REPO,
        repo_type="space",
        private=False,
        space_sdk=SPACE_SDK,
        token=HF_TOKEN
    )
    print(f"Created Space '{SPACE_REPO}'.")

api.upload_folder(
    folder_path=LOCAL_DEPLOYMENT_DIR,
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Deploy Streamlit app to Hugging Face Space"
)

print(f"Deployment files uploaded successfully to Space: {SPACE_REPO}")

Overwriting tourism_project/hosting/hosting.py


# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.


```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Upload dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Run data preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &
          sleep 5
      - name: Train and register model
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/train.py

  deploy-hosting:
    needs: [register-dataset, data-prep, model-training]
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Push deployment to Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/hosting/hosting.py
```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [45]:
%%writefile tourism_project/requirements.txt
pandas==2.2.2
scikit-learn==1.6.0

huggingface_hub==0.34.4
mlflow==3.0.1
joblib==1.4.2
fsspec>=2024.6.1

Overwriting tourism_project/requirements.txt


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [46]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "sachin2213105@iitgoa.ac.in"
!git config --global user.name "SachinYadav1996"

# Clone your GitHub repository
!git clone https://github.com/SachinYadav1996/visit-with-us-tourism-prediction.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/visit-with-us-tourism-prediction

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'visit-with-us-tourism-prediction'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 61 (delta 12), reused 33 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 158.64 KiB | 5.67 MiB/s, done.
Resolving deltas: 100% (12/12), done.
mv: cannot move '/content/tourism_project/' to '/content/visit-with-us-tourism-prediction/tourism_project': Directory not empty


In [47]:
# 1. Yahan apna GitHub token paste karein (quotes ke andar)
MY_TOKEN="YOUR_GITHUB_TOKEN_HERE"

# 2. Root directory mein wapas aayein
%cd /content/

# 3. Purane failed/incomplete folders ko clean karein (Safe state ke liye)
!rm -rf /content/visit-with-us-tourism-prediction/

# 4. Git clone karein (Isse aapki pipeline.yml file directly Colab mein aa jayegi)
!git clone https://$MY_TOKEN@github.com/SachinYadav1996/visit-with-us-tourism-prediction.git

# 5. Apna project folder safe tarike se naye clone ke andar COPY karein
# Dhyan dein: Hum folder ke andar ka content nahi, balki poora folder copy kar rahe hain
!cp -r /content/tourism_project /content/visit-with-us-tourism-prediction/

# 6. Repository folder ke andar jaayein
%cd /content/visit-with-us-tourism-prediction/

# 7. Git configuration set karein
!git config --global user.email "sachin2213105@iitgoa.ac.in"
!git config --global user.name "SachinYadav1996"

# 8. Files ko add aur commit karein
!git add .
!git commit -m "Uploaded tourism_project code to trigger and fix pipeline"

# 9. Successfully push karein
!git push https://$MY_TOKEN@github.com/SachinYadav1996/visit-with-us-tourism-prediction.git main

/content
Cloning into 'visit-with-us-tourism-prediction'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 61 (delta 12), reused 33 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 158.64 KiB | 5.67 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/visit-with-us-tourism-prediction
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

In [48]:
from IPython.display import Markdown, display

github_repo_link = "https://github.com/SachinYadav1996/visit-with-us-tourism-prediction"
display(Markdown(f"### GitHub Repository Link\n[{github_repo_link}]({github_repo_link})"))
display(Markdown("Attach screenshots below this section showing:\n1. Repository folder structure\n2. GitHub Actions workflow execution status"))

### GitHub Repository Link
[https://github.com/Nikidsouza/visit-with-us-tourism-prediction](https://github.com/Nikidsouza/visit-with-us-tourism-prediction)

Attach screenshots below this section showing:
1. Repository folder structure
2. GitHub Actions workflow execution status

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

In [40]:
from IPython.display import Markdown, display

hf_space_link = "https://huggingface.co/spaces/SachinY1996/visit-with-us-tourism-prediction"
display(Markdown(f"### Hugging Face Space Link\n[{hf_space_link}]({hf_space_link})"))
display(Markdown("Attach screenshots below this section showing:\n1. Streamlit app homepage\n2. Prediction example output"))

### Hugging Face Space Link
[https://huggingface.co/spaces/Nikidsouza23/visit-with-us-tourism-prediction](https://huggingface.co/spaces/Nikidsouza23/visit-with-us-tourism-prediction)

Attach screenshots below this section showing:
1. Streamlit app homepage
2. Prediction example output

<font size=6 color="navyblue">Power Ahead!</font>
___